# EP2 — Reconocimiento de letras ASL mediante una CNN

## Etapas 1 y 2: preparación de imágenes y diseño de la arquitectura CNN

**Asignatura:** Técnicas Avanzadas de Machine Learning I (TLY1102)  
**Dataset:** Sign Language MNIST (Kaggle)  
**Variante:** clasificación multiclase de las letras A, E y S.  
**Objetivo:** preparar datos reproducibles para entrenar una red neuronal convolucional (CNN), conservando la MLP de la EP1 como referencia.

### Problema
Las letras A, E y S tienen configuraciones de mano visualmente parecidas. Se pretende distinguirlas a partir de imágenes de 28 × 28 píxeles en escala de grises. Esto es un experimento académico sobre **ASL estadounidense**, no un traductor completo ni un sistema validado para la lengua de señas chilena (LSCh).

### Criterios de éxito propuestos, heredados de EP1
- Accuracy en test mayor a 90 %.
- F1 macro en test mayor a 0,85.
- Recall mayor a 0,80 para cada clase.
- Registrar y explicar las confusiones entre letras, no solo una puntuación global.

Las etapas 1 y 2 **preparan y diseñan una CNN; todavía no la entrenan ni evalúan**. Se reserva el test original sin utilizarlo para ajustar decisiones del modelo.

## 1. Importaciones, semilla y rutas de datos

El notebook funciona tanto si Jupyter se abre desde la raíz del repositorio como si se abre desde la carpeta notebooks. Los archivos CSV originales ya están almacenados en data/.

In [ ]:
from pathlib import Path
import hashlib
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TAMANO_IMAGEN = (28, 28)
CANALES = 1
LETRAS_OBJETIVO = ("A", "E", "S")
ETIQUETAS_ORIGINALES = {"A": 0, "E": 4, "S": 18}
ETIQUETA_A_INDICE = {0: 0, 4: 1, 18: 2}
INDICE_A_LETRA = {0: "A", 1: "E", 2: "S"}
N_CLASES = len(LETRAS_OBJETIVO)

CANDIDATAS_DATA = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path("/content/asl-cnn-aes/data"),
]

DATA_DIR = next(
    (ruta for ruta in CANDIDATAS_DATA
     if (ruta / "sign_mnist_train.csv").is_file()
     and (ruta / "sign_mnist_test.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "No se encontraron sign_mnist_train.csv y sign_mnist_test.csv. "
        "Ejecuta Jupyter desde la raíz o notebooks/ del repositorio, "
        "o clona asl-cnn-aes en /content/ si usas Google Colab."
    )

sns.set_theme(style="whitegrid")
print("Directorio de datos:", DATA_DIR.resolve())
print("Semilla:", SEED)

## 2. Inspección del dataset completo

Cada fila del CSV es una imagen: una columna label y 784 columnas pixel1…pixel784. Los píxeles originales están en el rango 0–255. Las 24 clases corresponden a las letras A–Y, sin J, y tampoco se incluye Z porque ambas requieren movimiento en ASL.

Comprobaremos la estructura real de los archivos antes de asumir tamaños, formatos y calidad de los datos.

In [ ]:
df_train_full = pd.read_csv(DATA_DIR / "sign_mnist_train.csv")
df_test_full = pd.read_csv(DATA_DIR / "sign_mnist_test.csv")

COLUMNAS_PIXELES = [f"pixel{i}" for i in range(1, 785)]
COLUMNAS_ESPERADAS = ["label", *COLUMNAS_PIXELES]
for nombre, df in [("Entrenamiento", df_train_full), ("Prueba", df_test_full)]:
    if df.columns.tolist() != COLUMNAS_ESPERADAS:
        raise ValueError(f"{nombre}: columnas inesperadas. Revisar formato del CSV.")
    if df.isna().to_numpy().any():
        raise ValueError(f"{nombre}: hay valores nulos.")
    minimo = df[COLUMNAS_PIXELES].to_numpy().min()
    maximo = df[COLUMNAS_PIXELES].to_numpy().max()
    if not (0 <= minimo <= maximo <= 255):
        raise ValueError(f"{nombre}: valores de píxeles fuera del rango 0–255.")
    print(
        f"{nombre}: {len(df):,} imágenes | {df.shape[1]} columnas "
        f"| clases={df['label'].nunique()} | píxeles [{minimo}, {maximo}] "
        f"| filas idénticas repetidas={df.duplicated().sum()}"
    )

if len(df_train_full) != 27455 or len(df_test_full) != 7172:
    print("Advertencia: los tamaños no coinciden con la versión del dataset usada en EP1.")

In [ ]:
# Mostrar distribución del conjunto de entrenamiento original (24 clases).
etiqueta_a_letra = {i: chr(ord("A") + i) for i in range(25) if i != 9}
conteo_completo = (
    df_train_full["label"].value_counts().sort_index().rename_axis("label")
    .reset_index(name="cantidad")
)
conteo_completo["letra"] = conteo_completo["label"].map(etiqueta_a_letra)
if conteo_completo["letra"].isna().any():
    raise ValueError("Se encontraron etiquetas sin correspondencia en el alfabeto ASL.")

plt.figure(figsize=(12, 4))
sns.barplot(data=conteo_completo, x="letra", y="cantidad", color="steelblue")
plt.title("Distribución de letras del dataset original (train)")
plt.xlabel("Letra ASL")
plt.ylabel("Cantidad de imágenes")
plt.tight_layout()
plt.show()

## 3. Variante A / E / S y análisis exploratorio

Elegimos tres señas estáticas con similitudes geométricas. La selección no demuestra por sí sola que sean las tres clases más difíciles del dataset; esa afirmación requeriría un experimento adicional. Para la EP2 mantenemos la misma variante de la EP1 para poder comparar modelos.

Se filtran las etiquetas originales 0 (A), 4 (E) y 18 (S). Las nuevas etiquetas serán 0, 1 y 2.

In [ ]:
def filtrar_clases(df: pd.DataFrame) -> pd.DataFrame:
    """Selecciona A/E/S sin modificar el dataset original."""
    filtrado = df.loc[df["label"].isin(ETIQUETA_A_INDICE)].copy()
    filtrado["clase"] = filtrado["label"].map(ETIQUETA_A_INDICE).astype("int64")
    return filtrado

df_train_aes = filtrar_clases(df_train_full)
df_test_aes = filtrar_clases(df_test_full)

resumen_clases = pd.DataFrame({
    "letra": list(LETRAS_OBJETIVO),
    "label_original": [ETIQUETAS_ORIGINALES[c] for c in LETRAS_OBJETIVO],
    "label_nuevo": list(range(N_CLASES)),
    "train_original": [
        int((df_train_aes["label"] == ETIQUETAS_ORIGINALES[c]).sum())
        for c in LETRAS_OBJETIVO
    ],
    "test_original": [
        int((df_test_aes["label"] == ETIQUETAS_ORIGINALES[c]).sum())
        for c in LETRAS_OBJETIVO
    ],
})
display(resumen_clases)
print("Total de imágenes A/E/S:", len(df_train_aes) + len(df_test_aes))

In [ ]:
# Visualización reproducible de ejemplos de cada clase.
fig, ejes = plt.subplots(N_CLASES, 4, figsize=(10, 8))
for fila, letra in enumerate(LETRAS_OBJETIVO):
    muestra = df_train_aes.loc[
        df_train_aes["label"] == ETIQUETAS_ORIGINALES[letra]
    ].sample(n=4, random_state=SEED)
    for columna, (_, registro) in enumerate(muestra.iterrows()):
        imagen = registro[COLUMNAS_PIXELES].to_numpy(dtype=np.uint8).reshape(TAMANO_IMAGEN)
        ejes[fila, columna].imshow(imagen, cmap="gray", vmin=0, vmax=255)
        ejes[fila, columna].set_title(f"Letra {letra}")
        ejes[fila, columna].axis("off")
fig.suptitle("Variabilidad visual de A, E y S", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Comparación del número de ejemplos disponibles para cada letra.
resumen_largo = resumen_clases.melt(
    id_vars=["letra"], value_vars=["train_original", "test_original"],
    var_name="particion", value_name="imagenes"
)
resumen_largo["particion"] = resumen_largo["particion"].map({
    "train_original": "Train original", "test_original": "Test original"
})
plt.figure(figsize=(7, 4))
sns.barplot(data=resumen_largo, x="letra", y="imagenes", hue="particion")
plt.title("Distribución de clases seleccionadas")
plt.ylabel("Cantidad de imágenes")
plt.xlabel("Letra ASL")
plt.tight_layout()
plt.show()

## 4. Separación, normalización y forma de entrada para CNN

Conservamos el test original aparte (1.075 imágenes A/E/S). Dividimos únicamente el train original, de forma estratificada, en 80 % entrenamiento y 20 % validación, con semilla 42. Este es el mismo criterio de partición usado en EP1 y permite replicar el experimento.

- La MLP de EP1 recibe vectores (784,).
- La CNN recibirá tensores (28, 28, 1), conservando el vecindario espacial entre píxeles.
- Normalizamos dividiendo los valores de intensidad por 255, sin ajustar parámetros utilizando test.
- Prepararemos también etiquetas one-hot, compatibles con una salida softmax y categorical_crossentropy.

**Precaución:** Sign Language MNIST incluye variaciones derivadas de fotografías de origen. Una división aleatoria estratificada puede repartir imágenes casi idénticas entre entrenamiento y validación. La auditoría de coincidencias exactas no detecta todas las casi duplicadas. Antes de interpretar la validación como evidencia fuerte de generalización, conviene definir grupos de imágenes de origen si pueden reconstruirse y, para una comparación más rigurosa, volver a entrenar ambos modelos bajo el mismo protocolo.

In [ ]:
# Convertir solo las columnas de píxeles, sin introducir las etiquetas en X.
X_train_full_plano = (
    df_train_aes[COLUMNAS_PIXELES].to_numpy(dtype=np.float32) / 255.0
)
y_train_full = df_train_aes["clase"].to_numpy(dtype=np.int64)

X_test_plano = df_test_aes[COLUMNAS_PIXELES].to_numpy(dtype=np.float32) / 255.0
y_test = df_test_aes["clase"].to_numpy(dtype=np.int64)

X_train_plano, X_val_plano, y_train, y_val = train_test_split(
    X_train_full_plano,
    y_train_full,
    test_size=0.20,
    stratify=y_train_full,
    random_state=SEED,
)

# Dimensión final de entrada: (n_imagenes, 28, 28, 1).
X_train = X_train_plano.reshape(-1, *TAMANO_IMAGEN, CANALES)
X_val = X_val_plano.reshape(-1, *TAMANO_IMAGEN, CANALES)
X_test = X_test_plano.reshape(-1, *TAMANO_IMAGEN, CANALES)

# One-hot para entrenar después una CNN con categorical_crossentropy.
y_train_onehot = np.eye(N_CLASES, dtype=np.float32)[y_train]
y_val_onehot = np.eye(N_CLASES, dtype=np.float32)[y_val]
y_test_onehot = np.eye(N_CLASES, dtype=np.float32)[y_test]

for nombre, x, y, oh in [
    ("Entrenamiento", X_train, y_train, y_train_onehot),
    ("Validación", X_val, y_val, y_val_onehot),
    ("Prueba", X_test, y_test, y_test_onehot),
]:
    print(f"{nombre}: X={x.shape}, y={y.shape}, one-hot={oh.shape}")
    print("  clases:", {INDICE_A_LETRA[i]: int((y == i).sum()) for i in range(N_CLASES)})

In [ ]:
# Pruebas automáticas mínimas del preprocesamiento.
assert X_train.shape == (2625, 28, 28, 1)
assert X_val.shape == (657, 28, 28, 1)
assert X_test.shape == (1075, 28, 28, 1)
assert X_train.dtype == X_val.dtype == X_test.dtype == np.float32
assert 0.0 <= X_train.min() <= X_train.max() <= 1.0
assert 0.0 <= X_val.min() <= X_val.max() <= 1.0
assert 0.0 <= X_test.min() <= X_test.max() <= 1.0
assert all(np.isfinite(x).all() for x in [X_train, X_val, X_test])
assert all(np.allclose(y.sum(axis=1), 1) for y in
           [y_train_onehot, y_val_onehot, y_test_onehot])
assert np.array_equal(np.unique(y_train), [0, 1, 2])
assert np.array_equal(np.unique(y_val), [0, 1, 2])
assert np.array_equal(np.unique(y_test), [0, 1, 2])
print("Comprobaciones de forma, tipo, normalización y etiquetas: OK.")

## 5. Auditoría preliminar de duplicados entre particiones

Buscamos coincidencias **exactas** entre imágenes de entrenamiento, validación y test. Esta comprobación ayuda a detectar fallas evidentes, pero **cero coincidencias exactas no implica ausencia de leakage**: variaciones por rotación, recorte o brillo pueden seguir siendo casi idénticas. Tampoco se usa el test para elegir hiperparámetros.

In [ ]:
def huellas_exactas(x: np.ndarray) -> set[bytes]:
    """Huella por imagen: solo detecta píxeles exactamente iguales."""
    return {
        hashlib.blake2b(imagen.tobytes(), digest_size=16).digest()
        for imagen in x
    }

h_train = huellas_exactas(X_train)
h_val = huellas_exactas(X_val)
h_test = huellas_exactas(X_test)

coinciden_train_val = len(h_train & h_val)
coinciden_train_test = len(h_train & h_test)
coinciden_val_test = len(h_val & h_test)
print("Coincidencias exactas entre train y validación:", coinciden_train_val)
print("Coincidencias exactas entre train y test:", coinciden_train_test)
print("Coincidencias exactas entre validación y test:", coinciden_val_test)
if coinciden_train_val:
    print("Advertencia: hay imágenes idénticas entre entrenamiento y validación.")
print("Nota: falta verificar similitud de imágenes aumentadas; esta auditoría es parcial.")

## 6. Resultado de la etapa 1 y siguiente etapa

Los tensores X_train, X_val y X_test (28 × 28 × 1) y sus etiquetas están preparados para conectar una CNN. Se conservan los CSV y el notebook MLP originales, sin sobrescribirlos.

**Decisión metodológica:** utilizar el conjunto test original solamente en la evaluación final. Reportar las limitaciones de validación por posibles variantes de fotografías originales; en la comparación MLP/CNN, explicitar que los modelos deben usar las mismas clases, particiones, métricas y equipo de entrenamiento, y que se deberá volver a medir el tiempo de entrenamiento de la MLP de manera comparable.

**Continuación:** la etapa 2 (a continuación) construye y justifica dos CNN antes de entrenarlas. La etapa 3 entrenará y comparará las curvas de aprendizaje; las métricas de test y Feature Maps vendrán después. No se afirma ninguna mejora de CNN hasta medirla.

---
## 7. Etapa 2 — Diseño de una Red Neuronal Convolucional (CNN)

La red recibirá imágenes de **28 × 28 × 1** (alto × ancho × canales), sin aplanarlas antes de las convoluciones.

**Flujo de la arquitectura**

1. `Conv2D` de 3×3, padding `same`, activación ReLU: detecta patrones locales (bordes, dedos y contornos) sin cambiar las dimensiones espaciales.
2. `MaxPooling2D(2×2)`: reduce a la mitad alto y ancho, disminuyendo cálculos y reteniendo las activaciones más fuertes de cada región.
3. Segunda `Conv2D` de 3×3 + `MaxPooling2D(2×2)`: aprende combinaciones de rasgos más complejos.
4. `Flatten` transforma los mapas resultantes (7×7×filtros) en un vector, ahora sí, para la clasificación.
5. `Dense(64, relu)` combina los rasgos; `Dense(3, softmax)` entrega probabilidades para A, E y S.

**Experimento de arquitectura:** se proponen dos configuraciones **iguales excepto por la cantidad de filtros**. Mantener el tamaño de kernel, la cantidad de capas, las neuronas densas y el learning rate permite evaluar el efecto del número de filtros. No se elige la mejor por el test: eso se hará usando únicamente validación.

La MLP anterior, por el contrario, recibe los 784 valores planos directamente, sin filtros espaciales compartidos.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibilidad de inicialización y operaciones aleatorias comunes.
# Algunas operaciones de GPU pueden seguir siendo no deterministas.
keras.utils.set_random_seed(SEED)

HIPERPARAMETROS = {
    "kernel_size": (3, 3),
    "pool_size": (2, 2),
    "padding": "same",
    "activacion_oculta": "relu",
    "unidades_densas": 64,
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs_max": 40,
    "funcion_perdida": "categorical_crossentropy",
    "optimizador": "Adam",
}
CONFIGURACIONES_CNN = {
    "cnn_base": (16, 32),
    "cnn_mas_filtros": (32, 64),
}

def construir_cnn(nombre_configuracion: str) -> keras.Model:
    """Crea y compila una CNN; NO la entrena ni examina el test."""
    if nombre_configuracion not in CONFIGURACIONES_CNN:
        raise ValueError(f"Configuración desconocida: {nombre_configuracion}")

    filtros_1, filtros_2 = CONFIGURACIONES_CNN[nombre_configuracion]
    keras.utils.set_random_seed(SEED)  # Reinicia la semilla antes de crear cada modelo.
    modelo = keras.Sequential([
        keras.Input(shape=(*TAMANO_IMAGEN, CANALES)),
        layers.Conv2D(
            filtros_1, HIPERPARAMETROS["kernel_size"],
            padding=HIPERPARAMETROS["padding"],
            activation=HIPERPARAMETROS["activacion_oculta"],
            name="conv_bordes",
        ),
        layers.MaxPooling2D(
            pool_size=HIPERPARAMETROS["pool_size"], name="pool_1"
        ),
        layers.Conv2D(
            filtros_2, HIPERPARAMETROS["kernel_size"],
            padding=HIPERPARAMETROS["padding"],
            activation=HIPERPARAMETROS["activacion_oculta"],
            name="conv_patrones",
        ),
        layers.MaxPooling2D(
            pool_size=HIPERPARAMETROS["pool_size"], name="pool_2"
        ),
        layers.Flatten(name="aplanado_final"),
        layers.Dense(
            HIPERPARAMETROS["unidades_densas"],
            activation=HIPERPARAMETROS["activacion_oculta"],
            name="densa_oculta",
        ),
        layers.Dense(N_CLASES, activation="softmax", name="salida_aes"),
    ], name=nombre_configuracion)

    modelo.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=HIPERPARAMETROS["learning_rate"]
        ),
        loss=HIPERPARAMETROS["funcion_perdida"],
        metrics=["accuracy"],
    )
    return modelo

print("TensorFlow:", tf.__version__)
print("Formas de los datos:", X_train.shape, X_val.shape, X_test.shape)
print("Configuraciones:", CONFIGURACIONES_CNN)


## 8. Resumen de arquitecturas y cantidad de parámetros

Vamos a revisar el número de parámetros entrenables y el tamaño de salida de cada CNN **antes de entrenarla**. El objetivo es comprobar que las arquitecturas son coherentes y que el modelo más grande no se está confundiendo con uno necesariamente mejor.

Como referencia histórica, la MLP de EP1 tenía **108.931 parámetros**. La comparación de parámetros es válida como descripción estructural, pero aún **no** como comparación de precisión o velocidad.

In [ ]:
# Construir dos modelos comparables con los mismos hiperparámetros generales.
modelos_cnn = {
    nombre: construir_cnn(nombre)
    for nombre in CONFIGURACIONES_CNN
}

resumen_arquitecturas = pd.DataFrame([
    {
        "modelo": nombre,
        "filtros_1": CONFIGURACIONES_CNN[nombre][0],
        "filtros_2": CONFIGURACIONES_CNN[nombre][1],
        "parametros": modelo.count_params(),
        "salida": str(modelo.output_shape),
    }
    for nombre, modelo in modelos_cnn.items()
])
display(resumen_arquitecturas)

for nombre, modelo in modelos_cnn.items():
    print(f"\n=== Arquitectura: {nombre} ===")
    modelo.summary()

assert all(m.output_shape == (None, N_CLASES) for m in modelos_cnn.values())
assert modelos_cnn["cnn_base"].count_params() < modelos_cnn["cnn_mas_filtros"].count_params()
print("\nAmbos modelos producen tres probabilidades de salida; estructura verificada.")


## 9. Comprobación de forma (sin entrenar)

La prueba siguiente entrega **cuatro imágenes de validación** a los modelos recién inicializados. Sirve para detectar incompatibilidades entre la forma de entrada y las capas, así como errores en la salida `softmax`.

**Importante:** esas probabilidades son aleatorias desde el punto de vista predictivo, porque no hubo entrenamiento. No deben reportarse como resultados ni interpretarse como accuracy.

In [ ]:
# Solo pruebas técnicas de compatibilidad; ningún modelo ha ejecutado fit().
lote_prueba_forma = X_val[:4]
for nombre, modelo in modelos_cnn.items():
    probabilidades = modelo.predict(lote_prueba_forma, verbose=0)
    assert probabilidades.shape == (4, N_CLASES)
    assert np.isfinite(probabilidades).all()
    assert np.all(probabilidades >= 0) and np.all(probabilidades <= 1)
    assert np.allclose(probabilidades.sum(axis=1), 1.0, atol=1e-5)
    print(
        f"{nombre}: salida={probabilidades.shape}, "
        f"parámetros={modelo.count_params():,}; softmax OK"
    )
print("Prueba de compatibilidad superada. Modelos aún SIN ENTRENAR.")


## 10. Justificación de decisiones y preparación para la etapa 3

| Componente | Elección | Razón |
|---|---|---|
| Entrada | 28×28×1 | Coincide con el dataset real, monocromático y normalizado. |
| Dos bloques convolucionales | 16/32 o 32/64 filtros | Permiten observar el efecto de aumentar la capacidad de extracción de rasgos. |
| Filtros 3×3 y padding same | Fijos entre variantes | Capturan relaciones locales sin cambiar ancho/alto en cada convolución. |
| ReLU | Convoluciones y Dense oculta | Introduce no linealidad y suele facilitar la optimización. |
| MaxPooling 2×2 | Después de cada convolución | Reduce mapas de 28→14→7 píxeles y costo computacional. |
| Flatten y Dense(64) | Clasificador final | Convierten los mapas en rasgos combinados para asignar clases. |
| Softmax de 3 neuronas | A, E, S | Entrega probabilidades que suman 1 para clases mutuamente excluyentes. |
| Categorical crossentropy | Con etiquetas one-hot | Penaliza la probabilidad baja asignada a la clase correcta. |
| Adam a 0,001 | Igual en ambas CNN | Optimizador adaptativo; punto de partida, sujeto a resultados de validación. |
| Batch 64; máximo 40 épocas | Plan inicial | Se utilizará EarlyStopping monitoreando `val_loss` en la etapa 3. |

**Precauciones metodológicas:**
- La CNN con más filtros puede sobreajustarse y consumir más recursos. No se presupone que sea superior.
- Nunca seleccionar el modelo usando `X_test`. Se elegirá según evidencia de **validación** y se evaluará en test solo al final.
- Nuestra validación aleatoria puede incluir variantes visuales de fotografías del mismo origen: las métricas de validación pueden resultar optimistas. El problema requiere análisis adicional.
- El entrenamiento, las curvas, los tiempos, los ejemplos mal clasificados, Feature Maps y la comparación rigurosa con la MLP quedan pendientes.

**Estado al concluir la etapa 2:** las arquitecturas están construidas, compiladas, documentadas y verificadas por forma de entrada/salida; **todavía no se reporta accuracy de CNN**.